# Time Series & PMPM Forecasting — Healthcare

Demonstrates **time series forecasting** (ARIMA-style) and **PMPM (per member per month)** cost/utilization forecasting on healthcare claims data. Aligns with JD: ARIMA, time series, PMPM, healthcare claims/cost/utilization, model evaluation (MAPE, RMSE).

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

DATA_DIR = Path.cwd().parent.parent / "data" / "raw" if Path.cwd().name == "python" else Path("data") / "raw"
if not (DATA_DIR / "claims.csv").exists():
    DATA_DIR = Path.cwd() / "data" / "raw"

## 1. Build member-month (PMPM) aggregates from claims

In [ ]:
claims = pd.read_csv(DATA_DIR / "claims.csv")
encounters = pd.read_csv(DATA_DIR / "encounters.csv", parse_dates=["admit_date", "discharge_date"])

claims["service_month"] = pd.to_datetime(claims["submitted_date"]).dt.to_period("M")
member_months = claims.groupby("service_month").agg(
    total_paid=("total_paid", "sum"),
    total_charge=("total_charge", "sum"),
    member_count=("patient_id", "nunique"),
    claim_count=("claim_id", "count"),
).reset_index()
member_months["service_month"] = member_months["service_month"].dt.to_timestamp()
member_months["pmpm_paid"] = member_months["total_paid"] / member_months["member_count"].replace(0, np.nan)
member_months["pmpm_charge"] = member_months["total_charge"] / member_months["member_count"].replace(0, np.nan)
member_months = member_months.sort_values("service_month").set_index("service_month")
ts_cost = member_months["pmpm_paid"].dropna()
ts_cost = ts_cost.asfreq("MS").ffill().bfill()
print(ts_cost.head(10))
print("...")
print(ts_cost.tail(5))

## 2. Train/test split and ARIMA-style forecasting

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing

n = len(ts_cost)
train_size = max(1, int(0.8 * n))
train, test = ts_cost.iloc[:train_size], ts_cost.iloc[train_size:]

model = SARIMAX(train, order=(1, 0, 1), seasonal_order=(0, 0, 0, 12), enforce_stationarity=False)
fit = model.fit(disp=False)
pred = fit.forecast(steps=len(test))

rmse = np.sqrt(mean_squared_error(test, pred))
mape = mean_absolute_percentage_error(test, pred) * 100
print(f"SARIMA(1,0,1) forecast — RMSE: {rmse:.2f}, MAPE: {mape:.2f}%")
pd.DataFrame({"actual": test.values, "predicted": pred.values}, index=test.index).round(2)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
ax.plot(train.index, train.values, label="Train", color="steelblue")
ax.plot(test.index, test.values, label="Actual", color="green")
ax.plot(test.index, pred.values, label="SARIMA forecast", color="coral", linestyle="--")
ax.set_title("PMPM cost: actual vs SARIMA forecast")
ax.set_ylabel("PMPM paid")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Simple exponential smoothing (alternative)

In [ ]:
hw = ExponentialSmoothing(train, seasonal_periods=12, trend="add", seasonal="add").fit()
pred_hw = hw.forecast(steps=len(test))
rmse_hw = np.sqrt(mean_squared_error(test, pred_hw))
mape_hw = mean_absolute_percentage_error(test, pred_hw) * 100
print(f"Holt-Winters — RMSE: {rmse_hw:.2f}, MAPE: {mape_hw:.2f}%")

## 4. Prophet (optional)

Prophet is a JD-relevant alternative for seasonal time series. Requires `pip install prophet`. Same PMPM series; evaluate with MAPE/RMSE.

In [ ]:
try:
    from prophet import Prophet
    df_prophet = train.reset_index().rename(columns={"service_month": "ds", "pmpm_paid": "y"})
    m = Prophet(yearly_seasonality=True).fit(df_prophet)
    future = test.reset_index().rename(columns={"service_month": "ds"})[["ds"]]
    pred_prophet = m.predict(future)["yhat"].values
    rmse_p = np.sqrt(mean_squared_error(test, pred_prophet))
    mape_p = mean_absolute_percentage_error(test, pred_prophet) * 100
    print(f"Prophet — RMSE: {rmse_p:.2f}, MAPE: {mape_p:.2f}%")
except ImportError:
    print("Prophet not installed. Run: pip install prophet")

## Summary

PMPM (per member per month) series built from claims; SARIMA and exponential smoothing used for forecasting; evaluation via RMSE and MAPE. **Alternatives:** Prophet (`pip install prophet`) is another option for seasonal/time series forecasting—fit on the same PMPM series and evaluate with MAPE/RMSE. Production use would add back-testing, hyperparameter tuning, and pipeline automation (e.g. Airflow, Databricks).